# SLAM-Based Mapping Robot (Perception + Kinematics)

## Overview
This notebook implements a standalone Python simulation for **Simultaneous Localization and Mapping (SLAM)**. Due to the constraints of not having ROS/Gazebo installed on the host system, this custom simulation fulfills the required learning outcomes by explicitly modeling robot kinematics, LiDAR sensor ray-casting, and an Occupancy Grid Mapping algorithm.

### Features
- **Differential Drive Kinematics:** Models the robot's movement and generates noisy odometry (localization error).
- **Environment and Sensor:** Simulates a 2D environment with static obstacles and casts rays to mimic a 2D LiDAR.
- **Occupancy Grid Mapping:** Updates a log-odds probabilistic grid based on sensor readings and estimated pose to create a map of the unknown environment.

### Learning Outcomes
- **Probabilistic Robotics:** Utilized log-odds formulation for the occupancy grid and Gaussian noise for the sensor/odometry models.
- **Sensor Fusion:** Combined noisy odometry (estimated pose) with LiDAR depth scans to estimate the environment structure.
- **Coordinate Frames:** Managed transformations between the local robot frame (relative angles for LiDAR) and the global map frame (Occupancy grid).

---
## 1. Imports

In [ ]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

%matplotlib inline

---
## 2. Environment

The `Environment` class manages the 2D ground-truth map and simulates a 2D LiDAR sensor via ray-casting.

- The world is discretized into an occupancy grid (`0` = free, `1` = obstacle).
- Outer walls and several inner rectangular obstacles are generated.
- `get_lidar_scan()` casts rays from the robot's true pose and returns distances to the nearest obstacles (with optional Gaussian noise).

In [ ]:
class Environment:
    def __init__(self, width=50, height=50, resolution=1.0):
        self.width = width
        self.height = height
        self.resolution = resolution
        
        # Ground truth grid: 0 is free, 1 is obstacle
        self.grid_x_size = int(width / resolution)
        self.grid_y_size = int(height / resolution)
        self.grid = np.zeros((self.grid_x_size, self.grid_y_size))
        
        self.generate_obstacles()
        
    def generate_obstacles(self):
        """Generates some random obstacles in the environment."""
        # Outer walls
        self.grid[0, :] = 1
        self.grid[-1, :] = 1
        self.grid[:, 0] = 1
        self.grid[:, -1] = 1
        
        # Add some inner walls / boxes
        self.grid[10:15, 10:30] = 1
        self.grid[30:45, 10:15] = 1
        self.grid[25:30, 35:45] = 1
        
        # Add a few random blocks
        for _ in range(5):
            cx = np.random.randint(5, self.grid_x_size - 10)
            cy = np.random.randint(5, self.grid_y_size - 10)
            w = np.random.randint(2, 6)
            h = np.random.randint(2, 6)
            self.grid[cx:cx+w, cy:cy+h] = 1
            
    def get_lidar_scan(self, pose, max_range=15.0, num_rays=36, fov=np.pi*2, noise_std=0.05):
        """
        Simulates a 2D LiDAR scan using ray casting.
        pose: (x, y, theta)
        """
        x, y, theta = pose
        rel_angles = np.linspace(-fov/2, fov/2, num_rays, endpoint=False)
        distances = np.full(num_rays, max_range)
        
        step_size = self.resolution * 0.5  # Step size for ray casting
        
        for i, rel_angle in enumerate(rel_angles):
            angle = theta + rel_angle
            dx = np.cos(angle) * step_size
            dy = np.sin(angle) * step_size
            
            rx, ry = x, y
            dist = 0.0
            
            while dist < max_range:
                rx += dx
                ry += dy
                dist += step_size
                
                # Check grid coordinates
                gx = int(rx / self.resolution)
                gy = int(ry / self.resolution)
                
                if gx < 0 or gx >= self.grid_x_size or gy < 0 or gy >= self.grid_y_size:
                    distances[i] = dist
                    break
                    
                if self.grid[gx, gy] == 1:
                    # Hit an obstacle
                    distances[i] = dist + np.random.normal(0, noise_std)
                    break
                    
        return rel_angles, distances
        
    def plot_ground_truth(self, ax):
        """Plots the ground truth grid."""
        ax.imshow(self.grid.T, cmap='Greys', origin='lower', 
                  extent=[0, self.width, 0, self.height], alpha=0.5)
        ax.set_title("Ground Truth Environment")

---
## 3. Robot (Differential Drive Kinematics)

The `Robot` class models a differential-drive robot:

- Maintains both a **true pose** (ground truth) and an **estimated pose** (noisy odometry).
- Noise parameters (`alpha1`–`alpha4`) control the odometry drift, simulating real-world sensor imperfection.
- `move(v, w)` applies velocities to the true state, then generates a noisy odometry reading for the estimated state.

In [ ]:
class Robot:
    def __init__(self, init_pose, dt=0.1):
        """
        init_pose: (x, y, theta)
        dt: time step
        """
        self.true_pose = np.array(init_pose, dtype=float)
        self.estimated_pose = np.array(init_pose, dtype=float)
        self.dt = dt
        
        # Noise parameters for odometry
        self.alpha1 = 0.05
        self.alpha2 = 0.01
        self.alpha3 = 0.05
        self.alpha4 = 0.01
        
        self.true_path = [np.copy(self.true_pose)]
        self.estimated_path = [np.copy(self.estimated_pose)]
        
    def move(self, v, w):
        """
        Moves the robot with linear velocity v and angular velocity w.
        Updates true pose and generates noisy odometry to update estimated pose.
        """
        # True kinematics update
        if w == 0:
            self.true_pose[0] += v * np.cos(self.true_pose[2]) * self.dt
            self.true_pose[1] += v * np.sin(self.true_pose[2]) * self.dt
        else:
            self.true_pose[0] += (-v/w) * np.sin(self.true_pose[2]) + (v/w) * np.sin(self.true_pose[2] + w * self.dt)
            self.true_pose[1] += (v/w) * np.cos(self.true_pose[2]) - (v/w) * np.cos(self.true_pose[2] + w * self.dt)
            self.true_pose[2] += w * self.dt
            
        self.true_pose[2] = self.normalize_angle(self.true_pose[2])
        self.true_path.append(np.copy(self.true_pose))
        
        # Noisy odometry generation
        v_hat = v + np.random.normal(0, np.sqrt(self.alpha1 * v**2 + self.alpha2 * w**2 + 1e-6))
        w_hat = w + np.random.normal(0, np.sqrt(self.alpha3 * v**2 + self.alpha4 * w**2 + 1e-6))
        
        # Odometry update
        if w_hat == 0:
            self.estimated_pose[0] += v_hat * np.cos(self.estimated_pose[2]) * self.dt
            self.estimated_pose[1] += v_hat * np.sin(self.estimated_pose[2]) * self.dt
        else:
            self.estimated_pose[0] += (-v_hat/w_hat) * np.sin(self.estimated_pose[2]) + (v_hat/w_hat) * np.sin(self.estimated_pose[2] + w_hat * self.dt)
            self.estimated_pose[1] += (v_hat/w_hat) * np.cos(self.estimated_pose[2]) - (v_hat/w_hat) * np.cos(self.estimated_pose[2] + w_hat * self.dt)
            self.estimated_pose[2] += w_hat * self.dt
            
        self.estimated_pose[2] = self.normalize_angle(self.estimated_pose[2])
        self.estimated_path.append(np.copy(self.estimated_pose))
        
        # Return odometry command for SLAM module
        return v_hat, w_hat
        
    def normalize_angle(self, angle):
        while angle > np.pi:
            angle -= 2.0 * np.pi
        while angle < -np.pi:
            angle += 2.0 * np.pi
        return angle
        
    def get_true_pose(self):
        return self.true_pose
        
    def plot_robot(self, ax, pose, color='blue', label='Robot'):
        """Plots a simple representation of the robot."""
        x, y, theta = pose
        
        # Draw robot body
        circle = patches.Circle((x, y), 0.5, edgecolor=color, facecolor='none', label=label)
        ax.add_patch(circle)
        
        # Draw orientation line
        dx = np.cos(theta) * 0.5
        dy = np.sin(theta) * 0.5
        ax.plot([x, x+dx], [y, y+dy], color=color)

---
## 4. Occupancy Grid SLAM

The `OccupancyGridMap` class implements occupancy grid mapping using the **log-odds** representation:

- Each cell stores a log-odds value starting at `0` (unknown / 50% probability).
- For each LiDAR ray, **Bresenham's line algorithm** traces the cells the ray passes through:
  - Intermediate cells are marked as *free* (log-odds decreased).
  - The endpoint cell (if the ray hit an obstacle) is marked as *occupied* (log-odds increased).
- `get_map_probabilities()` converts the log-odds grid to probabilities in `[0, 1]`.

In [ ]:
class OccupancyGridMap:
    def __init__(self, width=50, height=50, resolution=1.0):
        self.width = width
        self.height = height
        self.resolution = resolution
        
        self.grid_x_size = int(width / resolution)
        self.grid_y_size = int(height / resolution)
        
        # Log odds map initialized to 0 (unknown)
        self.log_odds = np.zeros((self.grid_x_size, self.grid_y_size))
        
        # Log odds constants
        self.l_occ = np.log(0.7 / 0.3)
        self.l_free = np.log(0.3 / 0.7)
        self.l_0 = 0.0
        
    def update(self, pose, angles, distances, max_range=15.0):
        """
        Updates the occupancy grid given a pose and a LiDAR scan.
        pose: (x, y, theta)
        """
        x, y, theta = pose
        
        for i, angle in enumerate(angles):
            dist = distances[i]
            
            # Global angle of the ray
            ray_angle = theta + angle
            
            # Start and end points of the ray
            x0, y0 = x, y
            
            # If the ray hit nothing, we consider it free up to max_range
            is_hit = dist < max_range
            
            # Endpoint
            x1 = x + dist * np.cos(ray_angle)
            y1 = y + dist * np.sin(ray_angle)
            
            # Bresenham's line algorithm to find all cells crossed by the ray
            cells = list(self.bresenham(x0, y0, x1, y1))
            
            # Update cells
            for i, (cx, cy) in enumerate(cells):
                if 0 <= cx < self.grid_x_size and 0 <= cy < self.grid_y_size:
                    if i == len(cells) - 1 and is_hit:
                        # Endpoint is occupied
                        self.log_odds[cx, cy] += self.l_occ
                    else:
                        # Intermediate points are free
                        self.log_odds[cx, cy] += self.l_free
                        
    def bresenham(self, x0, y0, x1, y1):
        """Yields integer coordinates crossed by a line from (x0, y0) to (x1, y1)."""
        x0 = int(x0 / self.resolution)
        y0 = int(y0 / self.resolution)
        x1 = int(x1 / self.resolution)
        y1 = int(y1 / self.resolution)
        
        dx = abs(x1 - x0)
        dy = abs(y1 - y0)
        x, y = x0, y0
        sx = -1 if x0 > x1 else 1
        sy = -1 if y0 > y1 else 1
        
        if dx > dy:
            err = dx / 2.0
            while x != x1:
                yield (x, y)
                err -= dy
                if err < 0:
                    y += sy
                    err += dx
                x += sx
        else:
            err = dy / 2.0
            while y != y1:
                yield (x, y)
                err -= dx
                if err < 0:
                    x += sx
                    err += dy
                y += sy
        yield (x, y)
        
    def get_map_probabilities(self):
        """Converts log odds to probabilities (0 to 1)."""
        return 1.0 - (1.0 / (1.0 + np.exp(self.log_odds)))

---
## 5. Simulation Setup

Initialize all components:
- A **50×50** grid environment with obstacles.
- A robot starting at position `(5, 5)` facing right.
- An occupancy grid map for SLAM.
- A set of **waypoints** defining the exploration path.

In [ ]:
np.random.seed(42)
os.makedirs('plots', exist_ok=True)

# Initialize components
env = Environment(width=50, height=50, resolution=1.0)
init_pose = [5.0, 5.0, 0.0]
robot = Robot(init_pose, dt=0.1)
slam = OccupancyGridMap(width=50, height=50, resolution=1.0)

# Simple predefined path (waypoints) for exploration
waypoints = [
    [45.0, 5.0], [45.0, 45.0], [5.0, 45.0], [5.0, 20.0],
    [30.0, 20.0], [30.0, 30.0], [15.0, 30.0], [15.0, 15.0]
]

print(f"Environment size: {env.width} x {env.height}")
print(f"Initial robot pose: {init_pose}")
print(f"Number of waypoints: {len(waypoints)}")

---
## 6. Run the SLAM Simulation

The simulation loop:
1. **Drive** the robot towards the next waypoint using a proportional controller.
2. **Scan** the environment with the simulated LiDAR from the robot's true pose.
3. **Update** the SLAM occupancy grid using the noisy estimated pose and the LiDAR scan.
4. **Record** both true and estimated paths, plus periodic map snapshots for animation.

> **Note:** In a full SLAM system (e.g., FastSLAM), scan matching would correct the pose. Here we use raw noisy odometry to demonstrate localization drift.

In [ ]:
current_wp_idx = 0
num_steps = 1500
snapshot_interval = 10  # capture a frame every N steps

true_path_x = []
true_path_y = []
est_path_x = []
est_path_y = []

# Storage for animation frames
frames = []

for step in range(num_steps):
    # 1. Drive robot towards waypoint
    if current_wp_idx < len(waypoints):
        tx, ty = waypoints[current_wp_idx]
        rx, ry, rtheta = robot.true_pose
        
        dx = tx - rx
        dy = ty - ry
        distance = np.sqrt(dx**2 + dy**2)
        
        if distance < 1.0:
            current_wp_idx += 1
            v, w = 0.0, 0.0
        else:
            target_theta = np.arctan2(dy, dx)
            angle_diff = robot.normalize_angle(target_theta - rtheta)
            
            # Simple proportional controller
            v = min(2.0, distance) if abs(angle_diff) < np.pi/4 else 0.0
            w = max(-1.0, min(1.0, angle_diff * 2.0))
    else:
        v, w = 0.0, 0.0
        
    # Move robot
    v_hat, w_hat = robot.move(v, w)
    
    # 2. Get LiDAR scan from True Pose
    rel_angles, distances = env.get_lidar_scan(robot.true_pose, num_rays=72)
    
    # 3. Update SLAM map using Estimated Pose (Odometry) and LiDAR scan
    slam.update(robot.estimated_pose, rel_angles, distances)
    
    # Record paths
    true_path_x.append(robot.true_pose[0])
    true_path_y.append(robot.true_pose[1])
    est_path_x.append(robot.estimated_pose[0])
    est_path_y.append(robot.estimated_pose[1])
    
    # Snapshot for animation
    if step % snapshot_interval == 0 or current_wp_idx >= len(waypoints):
        frames.append({
            'step': step,
            'map_probs': slam.get_map_probabilities().copy(),
            'true_path_x': list(true_path_x),
            'true_path_y': list(true_path_y),
            'est_path_x': list(est_path_x),
            'est_path_y': list(est_path_y),
            'true_pose': robot.true_pose.copy(),
            'est_pose': robot.estimated_pose.copy(),
        })
    
    if current_wp_idx >= len(waypoints):
        break

print(f"Simulation completed after {step + 1} steps.")
print(f"Waypoints reached: {current_wp_idx}/{len(waypoints)}")
print(f"Animation frames captured: {len(frames)}")

---
## 7. Animated SLAM Visualization

Watch the occupancy grid map being built in real time as the robot explores:
- **Left:** Ground truth environment with the true robot path (green).
- **Right:** SLAM occupancy grid being constructed with the noisy odometry path (blue).

Use the playback controls below the animation to play, pause, or scrub through the simulation.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
plt.close(fig)  # prevent static display; we show the animation instead

def animate(frame_idx):
    f = frames[frame_idx]
    ax1.clear()
    ax2.clear()
    
    # --- Left panel: Ground Truth ---
    env.plot_ground_truth(ax1)
    ax1.plot(f['true_path_x'], f['true_path_y'], 'g-', linewidth=1.5, label='True Path')
    robot.plot_robot(ax1, f['true_pose'], color='green')
    ax1.set_xlim(0, env.width)
    ax1.set_ylim(0, env.height)
    ax1.set_aspect('equal')
    ax1.legend(loc='upper left')
    
    # --- Right panel: SLAM Map ---
    ax2.imshow(f['map_probs'].T, cmap='Greys', origin='lower',
              extent=[0, env.width, 0, env.height], vmin=0, vmax=1)
    ax2.plot(f['est_path_x'], f['est_path_y'], 'b-', linewidth=1.5, label='Estimated Path (Odometry)')
    robot.plot_robot(ax2, f['est_pose'], color='blue', label='Est Robot')
    ax2.set_xlim(0, env.width)
    ax2.set_ylim(0, env.height)
    ax2.set_aspect('equal')
    ax2.set_title(f"Generated SLAM Map  (step {f['step']})")
    ax2.legend(loc='upper left')
    
    fig.suptitle('SLAM Simulation', fontsize=14, fontweight='bold')
    fig.tight_layout()

anim = FuncAnimation(fig, animate, frames=len(frames), interval=80, repeat=True)
HTML(anim.to_jshtml())

### 7.1 Final Map (Static)

The completed SLAM map alongside the ground truth for reference.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Plot Ground Truth
env.plot_ground_truth(ax1)
ax1.plot(true_path_x, true_path_y, 'g-', linewidth=1.5, label='True Path')
robot.plot_robot(ax1, robot.true_pose, color='green')
ax1.legend()

# Plot Estimated SLAM Map
map_probs = slam.get_map_probabilities()
ax2.imshow(map_probs.T, cmap='Greys', origin='lower',
           extent=[0, env.width, 0, env.height], vmin=0, vmax=1)
ax2.plot(est_path_x, est_path_y, 'b-', linewidth=1.5, label='Estimated Path (Odometry)')
robot.plot_robot(ax2, robot.estimated_pose, color='blue', label='Est Robot')
ax2.set_title("Generated SLAM Map")
ax2.legend()

plt.tight_layout()
plt.savefig('plots/final_map.png', dpi=150)
plt.show()

### 7.2 Path Comparison (True vs. Estimated)

Overlay both paths on the same axes to visualize the **odometry drift** over time.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

ax.plot(true_path_x, true_path_y, 'g-', linewidth=1.5, label='True Path')
ax.plot(est_path_x, est_path_y, 'b--', linewidth=1.5, label='Estimated Path (Odometry)')

# Mark waypoints
for i, wp in enumerate(waypoints):
    ax.plot(wp[0], wp[1], 'rx', markersize=10)
    ax.annotate(f'WP{i}', (wp[0]+0.5, wp[1]+0.5), fontsize=8, color='red')

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('True Path vs. Estimated Path (Odometry Drift)')
ax.legend()
ax.set_xlim(0, 50)
ax.set_ylim(0, 50)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8. Localization Error Evaluation

Compute the **Mean Squared Error (MSE)** between the true and estimated positions to quantify odometry drift.

In [ ]:
# Calculate MSE localization error
true_path_arr = np.array(robot.true_path)
est_path_arr = np.array(robot.estimated_path)

mse = np.mean((true_path_arr[:, :2] - est_path_arr[:, :2])**2)
rmse = np.sqrt(mse)

# Per-step position error
errors = np.sqrt(np.sum((true_path_arr[:, :2] - est_path_arr[:, :2])**2, axis=1))

print(f"Localization Error (MSE):  {mse:.4f}")
print(f"Localization Error (RMSE): {rmse:.4f}")
print(f"Max Position Error:        {errors.max():.4f}")
print(f"Final Position Error:      {errors[-1]:.4f}")

### 8.1 Localization Error Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(errors, 'r-', linewidth=1)
ax.set_xlabel('Time Step')
ax.set_ylabel('Position Error (Euclidean)')
ax.set_title('Localization Error Over Time')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()